# DSN Bootcamp Qualification Hackathon 2026 — Validation & Model Selection Report

## 1. Purpose

This document summarizes the work completed in `Notebooks/02_validation_baseline.ipynb`.

The objective was to establish a rigorous, leakage-safe validation framework, compare candidate algorithms, test feature hypotheses, perform targeted tuning, analyze errors, and freeze a final model configuration before full-data training and submission.

**Metric:** RMSE on the original `total_sales` scale. Lower is better.

---

## 2. Validation Protocol

- 5-fold `KFold`, `shuffle=True`, `random_state=42`.
- The same locked folds were used for all model and feature comparisons.
- Leaderboard feedback was not used as validation.
- `product_category_clean` was created by lowercasing/trimming `product_category`.
- `id` was excluded because it is unique.
- CatBoost categorical columns were explicitly declared.
- Missing categorical values were represented as `__MISSING__`.
- Target-derived features were only considered when constructed from training information within each fold.
- The final confirmation reproduced the champion using the same locked folds.

---

## 3. Baseline Algorithm Comparison

| Model | Mean CV RMSE |
|---|---:|
| Naive mean | 1697.7241 |
| Random Forest | 1276.3164 |
| Ridge | 1136.6424 |
| LightGBM | 1133.7387 |
| HistGradientBoosting | 1125.5758 |
| XGBoost | 1109.3365 |
| CatBoost — log target | 1112.2017 |
| **CatBoost — raw target** | **1085.9531** |

Tree boosting clearly outperformed the naive, linear, and Random Forest baselines. CatBoost on the raw target became the initial champion.

The log-target experiment was explicitly scored on original-scale RMSE and was rejected.

---

## 4. Feature Engineering Experiments

| Experiment | CV RMSE | Decision |
|---|---:|---|
| Product-relative price | 1085.0348 | Reject — gain too small |
| Weight missing flag | 1086.8995 | Reject |
| Product median weight | 1086.5230 | Reject |
| Explicit store × category | 1086.7950 | Reject |
| Product target encoding | 1185.4279 | Reject — poor and leakage-sensitive |
| Product price squared | 1075.3621 | Neutral/reject — CatBoost captures curvature |
| Store-price band | 1083.2568 | Reject |
| Store-relative price | 1075.8920 | Reject |

Feature engineering was hypothesis-driven rather than a bulk feature dump.

The repeated experiments showed that the existing CatBoost representation already captured most of the useful price/store/category interactions tested.

---

## 5. CatBoost Tuning

### Depth

Best tested depth was 4.

| Depth | CV RMSE |
|---:|---:|
| 4 | **1080.5855** |
| 5 | 1084.1052 |
| 6 | 1085.9531 |
| 7 | 1089.8798 |
| 8 | 1091.5018 |

### Iterations

With depth 4 and learning rate 0.05:

| Iterations | CV RMSE |
|---:|---:|
| 300 | **1075.7865** |
| 500 | 1080.5855 |
| 800 | 1086.5047 |
| 1000 | 1089.7414 |

More iterations hurt rather than helped, suggesting overfitting.

### Learning rate

The refined search identified `learning_rate=0.025` as the best tested value, producing **1075.3555 RMSE** before the later Ordered Boosting/L2 improvements.

### Ordered boosting

- Plain configuration: 1075.3555
- Ordered boosting: **1074.9861**

Decision: use `boosting_type="Ordered"`.

### L2 regularization

| `l2_leaf_reg` | CV RMSE |
|---:|---:|
| 1 | **1074.9159** |
| 3 | 1074.9861 |
| 5 | 1075.2883 |
| 10 | 1075.5594 |
| 20 | 1075.2896 |

Decision: use `l2_leaf_reg=1`; further L2 tuning was stopped because gains were tiny.

### Random strength

| `random_strength` | CV RMSE |
|---:|---:|
| 0.00 | 1076.1353 |
| 0.25 | 1075.9468 |
| 0.50 | 1075.0384 |
| **1.00** | **1074.9159** |
| 2.00 | 1076.2659 |

Decision: retain `random_strength=1.0`.

---

## 6. XGBoost and LightGBM Checks

### XGBoost

Depth testing strongly favored shallow trees:

- depth 3: 1096.9276
- depth 4: 1113.6015
- depth 5: 1129.4606
- depth 6: 1150.6766
- depth 7: 1160.5989

The best tested configuration after iteration tuning used depth 3, 200 estimators, learning rate 0.05, subsample 0.8, and colsample 0.8, with approximately **1082.36 CV RMSE**.

Further tuning was stopped because it remained behind CatBoost.

### LightGBM

The best tested configuration used `num_leaves=7` and `n_estimators=100`, with approximately **1081.99 CV RMSE**.

Further tuning was stopped because it remained behind CatBoost.

---

## 7. OOF Diversity and Blending

Strict OOF predictions were generated using the locked folds.

- CatBoost: 1075.5259 OOF RMSE
- XGBoost: 1082.5609
- LightGBM: 1082.1644

Residual correlations were extremely high:

- CatBoost / XGBoost: 0.9914
- CatBoost / LightGBM: 0.9939
- XGBoost / LightGBM: 0.9953

Best tested blend:

- 88% CatBoost
- 12% XGBoost
- 0% LightGBM
- OOF RMSE: **1075.3785**

The improvement was only about 0.15 RMSE over CatBoost alone.

**Decision:** CatBoost remains the primary model; ensemble complexity was not justified by the evidence.

---

## 8. Error Analysis

The champion's OOF predictions were analyzed by target decile, store, store × category, product frequency, and price within the high-sales tail.

### High-sales tail

- 90th-percentile cutoff: **4557.841**
- Tail observations: **682 / 6818 = 10%**
- Tail actual mean: **5818.53**
- Tail predicted mean: **3975.00**
- Tail contribution to total squared error: **40.60%**

The model systematically underpredicted the upper tail.

### Tail × price

High-tail observations were underpredicted across every price band. Mean underprediction remained roughly 1962 at the lower tail-price band and 1728 at the highest band.

**Inference:** price helps identify high-sales observations but does not explain the full upper-tail variation. Simple price transformations were therefore not pursued further.

### Store and store × category

High-volume stores generated much of the error because they contain many high-sales observations. No universal store bias or store × category correction was found.

### Product frequency

Extreme errors were not monotonically related to product observation count.

**Decision:** no product-frequency correction.

---

## 9. Tail/Regime Experiment

### Hypothesis

Because the top 10% contributed 40.60% of squared error, explicitly identifying and modeling the high-sales regime might improve RMSE.

### Design

- Tail cutoff: 4557.841
- Stage 1: CatBoost tail classifier
- Stage 2: separate CatBoost regressors for tail and non-tail
- Soft probability-weighted prediction
- Same locked 5-fold CV

### Result

- Two-stage CV RMSE: **1075.1865**
- Std: **22.0163**
- Champion: **1074.9159**

**Decision: Reject.** The regime model was 0.2706 RMSE worse and less stable.

The tail is genuinely difficult, but the evidence does not support a separately modeled regime with the available features.

---

## 10. Final Frozen Model

```text
CatBoostRegressor
boosting_type="Ordered"
iterations=300
learning_rate=0.025
depth=4
l2_leaf_reg=1
random_strength=1.0
loss_function="RMSE"
random_seed=42
```

Categorical features:

```text
product_code
fat_content
product_category_clean
store_code
store_size
store_location_tier
store_format
```

`id` is excluded.

### Final confirmation

The frozen model was retrained from scratch across the exact locked folds:

```text
Fold 1    1065.990810
Fold 2    1053.887357
Fold 3    1085.198997
Fold 4    1107.592529
Fold 5    1061.909675
```

Mean CV RMSE:

**1074.915874**

This exactly reproduced the frozen benchmark.

The final confirmation fold-level standard deviation was **21.590061**.


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# Preserve the cleaned category representation established during EDA
train["product_category_clean"] = (
    train["product_category"].str.strip().str.lower()
)

test["product_category_clean"] = (
    test["product_category"].str.strip().str.lower()
)

X = train.drop(columns="total_sales")
y = train["total_sales"]

# ID is an identifier, not a predictive feature
X = X.drop(columns="id")

# Locked CV folds — reused for every experiment
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(kf.split(X, y))

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of folds:", len(folds))

for i, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(
        f"Fold {i}: "
        f"train={len(train_idx)}, "
        f"validation={len(valid_idx)}"
    )

X shape: (6818, 12)
y shape: (6818,)
Number of folds: 5
Fold 1: train=5454, validation=1364
Fold 2: train=5454, validation=1364
Fold 3: train=5454, validation=1364
Fold 4: train=5455, validation=1363
Fold 5: train=5455, validation=1363


In [5]:
# Naive mean baseline

from sklearn.metrics import mean_squared_error
import numpy as np

fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    mean_prediction = np.full(
        len(y_valid),
        y_train.mean()
    )

    rmse = np.sqrt(
        mean_squared_error(y_valid, mean_prediction)
    )

    fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nMean CV RMSE:", np.mean(fold_rmse))
print("Std CV RMSE:", np.std(fold_rmse))

Fold 1: RMSE = 1716.8663
Fold 2: RMSE = 1651.8001
Fold 3: RMSE = 1685.7045
Fold 4: RMSE = 1722.1846
Fold 5: RMSE = 1712.0648

Mean CV RMSE: 1697.7240714332984
Std CV RMSE: 26.168240707436414


In [6]:
# Ridge baseline — raw target

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

numeric_features = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0)),
])

ridge_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    ridge_model.fit(X_train, y_train)

    predictions = ridge_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    ridge_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nRidge Mean CV RMSE:", np.mean(ridge_fold_rmse))
print("Ridge Std CV RMSE:", np.std(ridge_fold_rmse))
print(
    "Improvement vs naive:",
    np.mean(fold_rmse) - np.mean(ridge_fold_rmse)
)

Fold 1: RMSE = 1131.3343
Fold 2: RMSE = 1099.5975
Fold 3: RMSE = 1149.2786
Fold 4: RMSE = 1177.3026
Fold 5: RMSE = 1125.6992

Ridge Mean CV RMSE: 1136.6424317149786
Ridge Std CV RMSE: 25.819275144004955
Improvement vs naive: 561.0816397183198


In [7]:
# Random Forest baseline — raw target

from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

numeric_features = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

preprocessor_rf = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

rf_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    (
        "model",
        RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

rf_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    rf_model.fit(X_train, y_train)

    predictions = rf_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    rf_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nRandom Forest Mean CV RMSE:", np.mean(rf_fold_rmse))
print("Random Forest Std CV RMSE:", np.std(rf_fold_rmse))
print(
    "Improvement vs Ridge:",
    np.mean(ridge_fold_rmse) - np.mean(rf_fold_rmse)
)

Fold 1: RMSE = 1275.5630
Fold 2: RMSE = 1234.5164
Fold 3: RMSE = 1284.5309
Fold 4: RMSE = 1309.2630
Fold 5: RMSE = 1277.7085

Random Forest Mean CV RMSE: 1276.316354437589
Random Forest Std CV RMSE: 24.094385627378937
Improvement vs Ridge: -139.67392272261054


In [8]:
# HistGradientBoosting baseline — raw target

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

numeric_features = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

preprocessor_hgb = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "encoder",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1,
                    ),
                ),
            ]),
            categorical_features,
        ),
    ]
)

hgb_model = Pipeline([
    ("preprocessor", preprocessor_hgb),
    (
        "model",
        HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        ),
    ),
])

hgb_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    hgb_model.fit(X_train, y_train)

    predictions = hgb_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    hgb_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nHGB Mean CV RMSE:", np.mean(hgb_fold_rmse))
print("HGB Std CV RMSE:", np.std(hgb_fold_rmse))
print(
    "Improvement vs Ridge:",
    np.mean(ridge_fold_rmse) - np.mean(hgb_fold_rmse)
)

Fold 1: RMSE = 1129.7642
Fold 2: RMSE = 1093.5782
Fold 3: RMSE = 1129.2000
Fold 4: RMSE = 1145.9154
Fold 5: RMSE = 1129.4215

HGB Mean CV RMSE: 1125.5758475672874
HGB Std CV RMSE: 17.222139299187553
Improvement vs Ridge: 11.066584147691174


In [9]:
# XGBoost baseline — raw target

from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

numeric_features = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

preprocessor_xgb = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

xgb_model = Pipeline([
    ("preprocessor", preprocessor_xgb),
    (
        "model",
        XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

xgb_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    xgb_model.fit(X_train, y_train)

    predictions = xgb_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    xgb_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nXGBoost Mean CV RMSE:", np.mean(xgb_fold_rmse))
print("XGBoost Std CV RMSE:", np.std(xgb_fold_rmse))
print(
    "Improvement vs HGB:",
    np.mean(hgb_fold_rmse) - np.mean(xgb_fold_rmse)
)

Fold 1: RMSE = 1117.8845
Fold 2: RMSE = 1073.6605
Fold 3: RMSE = 1117.7821
Fold 4: RMSE = 1140.9445
Fold 5: RMSE = 1096.4108

XGBoost Mean CV RMSE: 1109.33645585329
XGBoost Std CV RMSE: 22.730203827142514
Improvement vs HGB: 16.239391713997293


In [10]:
# LightGBM baseline — raw target

from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

preprocessor_lgb = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

lgb_model = Pipeline([
    ("preprocessor", preprocessor_lgb),
    (
        "model",
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
])

lgb_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    lgb_model.fit(X_train, y_train)

    predictions = lgb_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    lgb_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nLightGBM Mean CV RMSE:", np.mean(lgb_fold_rmse))
print("LightGBM Std CV RMSE:", np.std(lgb_fold_rmse))
print(
    "Improvement vs XGBoost:",
    np.mean(xgb_fold_rmse) - np.mean(lgb_fold_rmse)
)

Fold 1: RMSE = 1139.6191
Fold 2: RMSE = 1091.7394
Fold 3: RMSE = 1150.9243
Fold 4: RMSE = 1156.4491
Fold 5: RMSE = 1129.9617

LightGBM Mean CV RMSE: 1133.7387157602159
LightGBM Std CV RMSE: 22.908053432663543
Improvement vs XGBoost: -24.402259906925792


In [12]:
# CatBoost baseline — raw target

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

X_cat = X.drop(columns="product_category").copy()
y_cat = y.copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

cat_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y_cat.iloc[train_idx]
    y_valid = y_cat.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_valid[col] = X_valid[col].fillna("Missing").astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(X_train, y_train, cat_features=cat_indices)
    predictions = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    cat_fold_rmse.append(rmse)
    print(f"Fold {fold}: RMSE = {rmse:.4f}")

cat_mean_rmse = np.mean(cat_fold_rmse)
cat_std_rmse = np.std(cat_fold_rmse)

print("\nCatBoost Mean CV RMSE:", cat_mean_rmse)
print("CatBoost Std CV RMSE:", cat_std_rmse)
print(
    "Improvement vs XGBoost:",
    np.mean(xgb_fold_rmse) - cat_mean_rmse,
)

Fold 1: RMSE = 1076.0201
Fold 2: RMSE = 1057.1375
Fold 3: RMSE = 1093.5818
Fold 4: RMSE = 1119.1931
Fold 5: RMSE = 1083.8328

CatBoost Mean CV RMSE: 1085.9530521487982
CatBoost Std CV RMSE: 20.47727697063577
Improvement vs XGBoost: 23.38340370449191


In [13]:
# CatBoost baseline — log1p target

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np


X_cat_log = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_log = np.log1p(train["total_sales"])

# Same categorical features as raw-target CatBoost
cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]


cat_log_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_cat_log.iloc[train_idx].copy()
    X_valid = X_cat_log.iloc[valid_idx].copy()

    y_train = y_log.iloc[train_idx]
    y_valid_original = train["total_sales"].iloc[valid_idx]

    for col in cat_features:
        X_train[col] = (
            X_train[col]
            .fillna("__MISSING__")
            .astype(str)
        )

        X_valid[col] = (
            X_valid[col]
            .fillna("__MISSING__")
            .astype(str)
        )

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    
    log_predictions = model.predict(X_valid)

    predictions = np.expm1(log_predictions)


    rmse = np.sqrt(
        mean_squared_error(
            y_valid_original,
            predictions
        )
    )

    cat_log_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")



cat_log_mean_rmse = np.mean(cat_log_fold_rmse)
cat_log_std_rmse = np.std(cat_log_fold_rmse)

print("\nCatBoost Log Target Mean CV RMSE:", cat_log_mean_rmse)
print("CatBoost Log Target Std CV RMSE:", cat_log_std_rmse)

print(
    "Improvement vs raw-target CatBoost:",
    np.mean(cat_fold_rmse) - cat_log_mean_rmse
)

Fold 1: RMSE = 1122.7506
Fold 2: RMSE = 1069.1057
Fold 3: RMSE = 1117.7689
Fold 4: RMSE = 1142.4232
Fold 5: RMSE = 1108.9599

CatBoost Log Target Mean CV RMSE: 1112.2016602094416
CatBoost Log Target Std CV RMSE: 24.180100691870457
Improvement vs raw-target CatBoost: -26.24860806064339


In [14]:
# Phase 4 — Feature Engineering Experiment 1

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

price_ratio_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_base.iloc[train_idx].copy()
    X_valid = X_base.iloc[valid_idx].copy()

    y_train = y_base.iloc[train_idx]
    y_valid = y_base.iloc[valid_idx]



    product_median_price = (
        X_train
        .groupby("product_code")["product_price"]
        .median()
    )

    global_median_price = X_train["product_price"].median()



    train_product_median = (
        X_train["product_code"]
        .map(product_median_price)
        .fillna(global_median_price)
    )

    valid_product_median = (
        X_valid["product_code"]
        .map(product_median_price)
        .fillna(global_median_price)
    )



    X_train["product_relative_price"] = (
        X_train["product_price"] /
        train_product_median
    )

    X_valid["product_relative_price"] = (
        X_valid["product_price"] /
        valid_product_median
    )



    for col in cat_features:
        X_train[col] = (
            X_train[col]
            .fillna("__MISSING__")
            .astype(str)
        )

        X_valid[col] = (
            X_valid[col]
            .fillna("__MISSING__")
            .astype(str)
        )

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]



    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    predictions = model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    price_ratio_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")


price_ratio_mean_rmse = np.mean(price_ratio_fold_rmse)
price_ratio_std_rmse = np.std(price_ratio_fold_rmse)

print("\nProduct-relative price CatBoost RMSE:", price_ratio_mean_rmse)
print("Product-relative price CatBoost Std:", price_ratio_std_rmse)

print(
    "Change vs current champion:",
    np.mean(cat_fold_rmse) - price_ratio_mean_rmse
)

Fold 1: RMSE = 1072.9828
Fold 2: RMSE = 1060.7495
Fold 3: RMSE = 1097.4152
Fold 4: RMSE = 1112.3106
Fold 5: RMSE = 1081.7160

Product-relative price CatBoost RMSE: 1085.0348410841539
Product-relative price CatBoost Std: 18.12888945212419
Change vs current champion: 0.9182110646443107


In [15]:
X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

weight_flag_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_base.iloc[train_idx].copy()
    X_valid = X_base.iloc[valid_idx].copy()

    y_train = y_base.iloc[train_idx]
    y_valid = y_base.iloc[valid_idx]

    # One hypothesis: explicitly expose whether product weight is missing.
    X_train["weight_missing_flag"] = (
        X_train["product_weight_kg"].isna().astype(int)
    )

    X_valid["weight_missing_flag"] = (
        X_valid["product_weight_kg"].isna().astype(int)
    )

    # CatBoost categorical handling
    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    predictions = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    weight_flag_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

weight_flag_mean_rmse = np.mean(weight_flag_fold_rmse)
weight_flag_std_rmse = np.std(weight_flag_fold_rmse)

print("\nWeight-missing flag CatBoost RMSE:", weight_flag_mean_rmse)
print("Weight-missing flag CatBoost Std:", weight_flag_std_rmse)
print(
    "Change vs current champion:",
    np.mean(cat_fold_rmse) - weight_flag_mean_rmse
)

Fold 1: RMSE = 1081.5458
Fold 2: RMSE = 1058.7222
Fold 3: RMSE = 1096.8877
Fold 4: RMSE = 1117.3637
Fold 5: RMSE = 1079.9781

Weight-missing flag CatBoost RMSE: 1086.8994986917992
Weight-missing flag CatBoost Std: 19.4851922289055
Change vs current champion: -0.9464465430010023


In [16]:
X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

product_weight_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_base.iloc[train_idx].copy()
    X_valid = X_base.iloc[valid_idx].copy()

    y_train = y_base.iloc[train_idx]
    y_valid = y_base.iloc[valid_idx]

    # Product's typical weight learned from TRAINING FOLD ONLY
    product_median_weight = (
        X_train.groupby("product_code")["product_weight_kg"].median()
    )

    global_median_weight = X_train["product_weight_kg"].median()

    train_product_weight = (
        X_train["product_code"]
        .map(product_median_weight)
        .fillna(global_median_weight)
    )

    valid_product_weight = (
        X_valid["product_code"]
        .map(product_median_weight)
        .fillna(global_median_weight)
    )

    # Add leakage-safe product-level typical weight
    X_train["product_median_weight"] = train_product_weight
    X_valid["product_median_weight"] = valid_product_weight

    # CatBoost categorical handling
    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    predictions = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    product_weight_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

product_weight_mean_rmse = np.mean(product_weight_fold_rmse)
product_weight_std_rmse = np.std(product_weight_fold_rmse)

print("\nProduct-median weight CatBoost RMSE:", product_weight_mean_rmse)
print("Product-median weight CatBoost Std:", product_weight_std_rmse)
print(
    "Change vs current champion:",
    np.mean(cat_fold_rmse) - product_weight_mean_rmse
)

Fold 1: RMSE = 1080.6905
Fold 2: RMSE = 1056.2755
Fold 3: RMSE = 1096.9270
Fold 4: RMSE = 1115.5934
Fold 5: RMSE = 1083.1285

Product-median weight CatBoost RMSE: 1086.522974645347
Product-median weight CatBoost Std: 19.564155266291344
Change vs current champion: -0.5699224965487701


In [17]:
X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
    "store_category",
]

store_category_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_base.iloc[train_idx].copy()
    X_valid = X_base.iloc[valid_idx].copy()

    y_train = y_base.iloc[train_idx]
    y_valid = y_base.iloc[valid_idx]

    # Explicit store × category interaction
    X_train["store_category"] = (
        X_train["store_code"].astype(str)
        + "__"
        + X_train["product_category_clean"].astype(str)
    )

    X_valid["store_category"] = (
        X_valid["store_code"].astype(str)
        + "__"
        + X_valid["product_category_clean"].astype(str)
    )

    # CatBoost categorical handling
    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    predictions = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    store_category_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

store_category_mean_rmse = np.mean(store_category_fold_rmse)
store_category_std_rmse = np.std(store_category_fold_rmse)

print("\nStore x category CatBoost RMSE:", store_category_mean_rmse)
print("Store x category CatBoost Std:", store_category_std_rmse)
print(
    "Change vs current champion:",
    np.mean(cat_fold_rmse) - store_category_mean_rmse
)

Fold 1: RMSE = 1076.5959
Fold 2: RMSE = 1058.2771
Fold 3: RMSE = 1098.0173
Fold 4: RMSE = 1117.4426
Fold 5: RMSE = 1083.6420

Store × category CatBoost RMSE: 1086.7949722436883
Store × category CatBoost Std: 19.962225394272565
Change vs current champion: -0.8419200948901562


In [18]:
depth_results = {}

X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for depth in [4, 5, 7, 8]:

    fold_rmse = []

    print(f"\nCatBoost depth={depth}:")

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

        X_train = X_base.iloc[train_idx].copy()
        X_valid = X_base.iloc[valid_idx].copy()

        y_train = y_base.iloc[train_idx]
        y_valid = y_base.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = (
                X_train[col]
                .fillna("Missing")
                .astype(str)
            )
            X_valid[col] = (
                X_valid[col]
                .fillna("Missing")
                .astype(str)
            )

        cat_indices = [
            X_train.columns.get_loc(col)
            for col in cat_features
        ]

        model = CatBoostRegressor(
            iterations=500,
            learning_rate=0.05,
            depth=depth,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_indices,
        )

        predictions = model.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(y_valid, predictions)
        )

        fold_rmse.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.4f}")

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    depth_results[depth] = {
        "fold_rmse": fold_rmse,
        "mean_rmse": mean_rmse,
        "std_rmse": std_rmse,
    }

    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Std RMSE:  {std_rmse:.4f}")

print("\nDepth Summary")

for depth, result in depth_results.items():
    print(
        f"Depth {depth}: "
        f"RMSE = {result['mean_rmse']:.4f}, "
        f"Std = {result['std_rmse']:.4f}, "
        f"Change vs depth 6 = "
        f"{np.mean(cat_fold_rmse) - result['mean_rmse']:.4f}"
    )


===== CatBoost depth=4 =====
Fold 1: RMSE = 1070.6806
Fold 2: RMSE = 1052.7069
Fold 3: RMSE = 1094.0811
Fold 4: RMSE = 1113.1884
Fold 5: RMSE = 1072.2703
Mean RMSE: 1080.5855
Std RMSE:  20.9263

===== CatBoost depth=5 =====
Fold 1: RMSE = 1076.7280
Fold 2: RMSE = 1055.2974
Fold 3: RMSE = 1093.7923
Fold 4: RMSE = 1114.0312
Fold 5: RMSE = 1080.6770
Mean RMSE: 1084.1052
Std RMSE:  19.4189

===== CatBoost depth=7 =====
Fold 1: RMSE = 1079.8642
Fold 2: RMSE = 1059.5618
Fold 3: RMSE = 1105.5791
Fold 4: RMSE = 1123.5004
Fold 5: RMSE = 1080.8933
Mean RMSE: 1089.8798
Std RMSE:  22.2579

===== CatBoost depth=8 =====
Fold 1: RMSE = 1084.8218
Fold 2: RMSE = 1055.2788
Fold 3: RMSE = 1105.4645
Fold 4: RMSE = 1124.0274
Fold 5: RMSE = 1087.9165
Mean RMSE: 1091.5018
Std RMSE:  22.9017

===== DEPTH SUMMARY =====
Depth 4: RMSE = 1080.5855, Std = 20.9263, Δ vs depth 6 = 5.3676
Depth 5: RMSE = 1084.1052, Std = 19.4189, Δ vs depth 6 = 1.8479
Depth 7: RMSE = 1089.8798, Std = 22.2579, Δ vs depth 6 = -3.9267


In [19]:
iteration_results = {}

X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for iterations in [300, 500, 800, 1000]:

    fold_rmse = []

    print(f"\nCatBoost iterations={iterations}, depth=4: ")

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

        X_train = X_base.iloc[train_idx].copy()
        X_valid = X_base.iloc[valid_idx].copy()

        y_train = y_base.iloc[train_idx]
        y_valid = y_base.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = (
                X_train[col]
                .fillna("Missing")
                .astype(str)
            )
            X_valid[col] = (
                X_valid[col]
                .fillna("Missing")
                .astype(str)
            )

        cat_indices = [
            X_train.columns.get_loc(col)
            for col in cat_features
        ]

        model = CatBoostRegressor(
            iterations=iterations,
            learning_rate=0.05,
            depth=4,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_indices,
        )

        predictions = model.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(y_valid, predictions)
        )

        fold_rmse.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.4f}")

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    iteration_results[iterations] = {
        "fold_rmse": fold_rmse,
        "mean_rmse": mean_rmse,
        "std_rmse": std_rmse,
    }

    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Std RMSE:  {std_rmse:.4f}")

print("\n Iteration Summary ")

for iterations, result in iteration_results.items():
    print(
        f"{iterations} iterations: "
        f"RMSE = {result['mean_rmse']:.4f}, "
        f"Std = {result['std_rmse']:.4f}, "
        f"Change vs depth-4 champion = "
        f"{1080.5855 - result['mean_rmse']:.4f}"
    )


===== CatBoost iterations=300, depth=4 =====
Fold 1: RMSE = 1065.9036
Fold 2: RMSE = 1049.3347
Fold 3: RMSE = 1089.6209
Fold 4: RMSE = 1109.3491
Fold 5: RMSE = 1064.7241
Mean RMSE: 1075.7865
Std RMSE:  21.1546

===== CatBoost iterations=500, depth=4 =====
Fold 1: RMSE = 1070.6806
Fold 2: RMSE = 1052.7069
Fold 3: RMSE = 1094.0811
Fold 4: RMSE = 1113.1884
Fold 5: RMSE = 1072.2703
Mean RMSE: 1080.5855
Std RMSE:  20.9263

===== CatBoost iterations=800, depth=4 =====
Fold 1: RMSE = 1079.1846
Fold 2: RMSE = 1055.8536
Fold 3: RMSE = 1097.3718
Fold 4: RMSE = 1118.7869
Fold 5: RMSE = 1081.3266
Mean RMSE: 1086.5047
Std RMSE:  20.8812

===== CatBoost iterations=1000, depth=4 =====
Fold 1: RMSE = 1082.0348
Fold 2: RMSE = 1057.6134
Fold 3: RMSE = 1100.8600
Fold 4: RMSE = 1122.6104
Fold 5: RMSE = 1085.5882
Mean RMSE: 1089.7414
Std RMSE:  21.5074

===== ITERATION SUMMARY =====
300 iterations: RMSE = 1075.7865, Std = 21.1546, Δ vs depth-4 champion = 4.7990
500 iterations: RMSE = 1080.5855, Std = 20.9

In [20]:
# Test different CatBoost learning rates while keeping depth=4 and iterations=300 fixed.

learning_rate_results = {}

X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for learning_rate in [0.02, 0.03, 0.05, 0.08, 0.10]:

    fold_rmse = []

    print(
        f"\n CatBoost "
        f"learning_rate={learning_rate}, "
        f"depth=4, iterations=300 :"
    )

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

        X_train = X_base.iloc[train_idx].copy()
        X_valid = X_base.iloc[valid_idx].copy()

        y_train = y_base.iloc[train_idx]
        y_valid = y_base.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = (
                X_train[col]
                .fillna("Missing")
                .astype(str)
            )
            X_valid[col] = (
                X_valid[col]
                .fillna("Missing")
                .astype(str)
            )

        cat_indices = [
            X_train.columns.get_loc(col)
            for col in cat_features
        ]

        model = CatBoostRegressor(
            iterations=300,
            learning_rate=learning_rate,
            depth=4,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_indices,
        )

        predictions = model.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(y_valid, predictions)
        )

        fold_rmse.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.4f}")

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    learning_rate_results[learning_rate] = {
        "fold_rmse": fold_rmse,
        "mean_rmse": mean_rmse,
        "std_rmse": std_rmse,
    }

    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Std RMSE:  {std_rmse:.4f}")

print("\n learning Rate Summary ")

for learning_rate, result in learning_rate_results.items():
    improvement = 1075.7865 - result["mean_rmse"]

    print(
        f"Learning rate {learning_rate}: "
        f"RMSE = {result['mean_rmse']:.4f}, "
        f"Std = {result['std_rmse']:.4f}, "
        f"RMSE improvement vs champion = {improvement:.4f}"
    )


===== CatBoost learning_rate=0.02, depth=4, iterations=300 =====
Fold 1: RMSE = 1067.4140
Fold 2: RMSE = 1050.8510
Fold 3: RMSE = 1086.8734
Fold 4: RMSE = 1106.8100
Fold 5: RMSE = 1065.5032
Mean RMSE: 1075.4903
Std RMSE:  19.4039

===== CatBoost learning_rate=0.03, depth=4, iterations=300 =====
Fold 1: RMSE = 1065.3625
Fold 2: RMSE = 1052.0489
Fold 3: RMSE = 1086.4921
Fold 4: RMSE = 1108.2137
Fold 5: RMSE = 1065.6916
Mean RMSE: 1075.5618
Std RMSE:  19.6973

===== CatBoost learning_rate=0.05, depth=4, iterations=300 =====
Fold 1: RMSE = 1065.9036
Fold 2: RMSE = 1049.3347
Fold 3: RMSE = 1089.6209
Fold 4: RMSE = 1109.3491
Fold 5: RMSE = 1064.7241
Mean RMSE: 1075.7865
Std RMSE:  21.1546

===== CatBoost learning_rate=0.08, depth=4, iterations=300 =====
Fold 1: RMSE = 1073.9379
Fold 2: RMSE = 1051.7724
Fold 3: RMSE = 1092.0762
Fold 4: RMSE = 1115.8953
Fold 5: RMSE = 1071.6629
Mean RMSE: 1081.0689
Std RMSE:  21.5946

===== CatBoost learning_rate=0.1, depth=4, iterations=300 =====
Fold 1: RMS

In [21]:
# Refine the learning-rate search around the current best region while keeping depth=4 and iterations=300 fixed.

learning_rate_refined_results = {}

for learning_rate in [0.01, 0.015, 0.02, 0.025, 0.03]:

    fold_rmse = []

    print(
        f"\nCatBoost "
        f"learning_rate={learning_rate}, "
        f"depth=4, iterations=300"
    )

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

        X_train = X_base.iloc[train_idx].copy()
        X_valid = X_base.iloc[valid_idx].copy()

        y_train = y_base.iloc[train_idx]
        y_valid = y_base.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = (
                X_train[col]
                .fillna("Missing")
                .astype(str)
            )
            X_valid[col] = (
                X_valid[col]
                .fillna("Missing")
                .astype(str)
            )

        cat_indices = [
            X_train.columns.get_loc(col)
            for col in cat_features
        ]

        model = CatBoostRegressor(
            iterations=300,
            learning_rate=learning_rate,
            depth=4,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_indices,
        )

        predictions = model.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(y_valid, predictions)
        )

        fold_rmse.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.4f}")

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    learning_rate_refined_results[learning_rate] = {
        "fold_rmse": fold_rmse,
        "mean_rmse": mean_rmse,
        "std_rmse": std_rmse,
    }

    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Std RMSE:  {std_rmse:.4f}")

print("\n Refined Learning Rate Summary")

for learning_rate, result in learning_rate_refined_results.items():

    improvement = 1075.7865 - result["mean_rmse"]

    print(
        f"Learning rate {learning_rate}: "
        f"RMSE = {result['mean_rmse']:.4f}, "
        f"Std = {result['std_rmse']:.4f}, "
        f"RMSE improvement vs previous champion = {improvement:.4f}"
    )


===== CatBoost learning_rate=0.01, depth=4, iterations=300 =====
Fold 1: RMSE = 1080.3234
Fold 2: RMSE = 1058.4938
Fold 3: RMSE = 1092.4838
Fold 4: RMSE = 1116.1349
Fold 5: RMSE = 1075.5455
Mean RMSE: 1084.5963
Std RMSE:  19.1750

===== CatBoost learning_rate=0.015, depth=4, iterations=300 =====
Fold 1: RMSE = 1067.9981
Fold 2: RMSE = 1052.2464
Fold 3: RMSE = 1086.7534
Fold 4: RMSE = 1107.5217
Fold 5: RMSE = 1067.6385
Mean RMSE: 1076.4316
Std RMSE:  19.0079

===== CatBoost learning_rate=0.02, depth=4, iterations=300 =====
Fold 1: RMSE = 1067.4140
Fold 2: RMSE = 1050.8510
Fold 3: RMSE = 1086.8734
Fold 4: RMSE = 1106.8100
Fold 5: RMSE = 1065.5032
Mean RMSE: 1075.4903
Std RMSE:  19.4039

===== CatBoost learning_rate=0.025, depth=4, iterations=300 =====
Fold 1: RMSE = 1066.5354
Fold 2: RMSE = 1051.4447
Fold 3: RMSE = 1086.9416
Fold 4: RMSE = 1106.6580
Fold 5: RMSE = 1065.1977
Mean RMSE: 1075.3555
Std RMSE:  19.3225

===== CatBoost learning_rate=0.03, depth=4, iterations=300 =====
Fold 1: 

In [25]:
# Aim: Test whether shallower XGBoost trees improve cross-validated RMSE while keeping all other settings fixed.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

numeric_features = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

depth_results = []

for depth in [3, 4, 5, 6, 7]:
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features,
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]),
                    categorical_features,
                ),
            ]
        )

        model = XGBRegressor(
            n_estimators=500,
            max_depth=depth,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                predictions,
            )
        )

        fold_scores.append(rmse)

    depth_results.append({
        "max_depth": depth,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

xgb_depth_results = pd.DataFrame(depth_results)

xgb_depth_results.sort_values("mean_rmse")

,max_depth,mean_rmse,std_rmse
0,3,1096.927586,18.839946
1,4,1113.601502,19.746073
2,5,1129.460619,20.480119
3,6,1150.676587,18.500571
4,7,1160.598936,17.471380


In [26]:
# Test whether changing XGBoost's number of boosting rounds improves CV RMSE while keeping its best tree depth and all other settings fixed.

estimator_results = []

for n_estimators in [200, 300, 400, 500, 700]:
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features,
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]),
                    categorical_features,
                ),
            ]
        )

        model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                predictions,
            )
        )

        fold_scores.append(rmse)

    estimator_results.append({
        "n_estimators": n_estimators,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

xgb_estimator_results = pd.DataFrame(estimator_results)

xgb_estimator_results.sort_values("mean_rmse")

,n_estimators,mean_rmse,std_rmse
0,200,1082.360095,20.994904
1,300,1087.020402,20.828558
2,400,1092.577716,19.844216
3,500,1096.927586,18.839946
4,700,1104.836639,18.904823


In [27]:
# Test whether LightGBM tree complexity, controlled by num_leaves, can substantially improve its cross-validated RMSE.

from lightgbm import LGBMRegressor

lgbm_leaf_results = []

for num_leaves in [7, 15, 31, 63]:
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features,
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]),
                    categorical_features,
                ),
            ]
        )

        model = LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=num_leaves,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="regression",
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                predictions,
            )
        )

        fold_scores.append(rmse)

    lgbm_leaf_results.append({
        "num_leaves": num_leaves,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

lgbm_leaf_results = pd.DataFrame(lgbm_leaf_results)

lgbm_leaf_results.sort_values("mean_rmse")

,num_leaves,mean_rmse,std_rmse
0,7,1096.108872,18.585480
1,15,1115.622266,21.740217
2,31,1138.483697,22.593714
3,63,1165.735000,26.843626


In [28]:
# Test whether fewer LightGBM boosting rounds improve generalization after identifying num_leaves=7 as the best tree complexity.

lgbm_estimator_results = []

for n_estimators in [100, 200, 300, 400, 500]:
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features,
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]),
                    categorical_features,
                ),
            ]
        )

        model = LGBMRegressor(
            n_estimators=n_estimators,
            learning_rate=0.05,
            num_leaves=7,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="regression",
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                predictions,
            )
        )

        fold_scores.append(rmse)

    lgbm_estimator_results.append({
        "n_estimators": n_estimators,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

lgbm_estimator_results = pd.DataFrame(lgbm_estimator_results)

lgbm_estimator_results.sort_values("mean_rmse")

,n_estimators,mean_rmse,std_rmse
0,100,1081.988825,19.648770
1,200,1085.432718,19.914547
2,300,1089.717055,19.639357
3,400,1092.876336,18.778516
4,500,1096.108872,18.585480


In [29]:
# Generate leakage-safe out-of-fold predictions for our strongest models and measure whether their errors are sufficiently different to justify blending.

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error

oof_predictions = {
    "catboost": np.zeros(len(y)),
    "xgboost": np.zeros(len(y)),
    "lightgbm": np.zeros(len(y)),
}

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    
    # CatBoost 
    X_train_cat = X_train.drop(columns=["product_category"]).copy()
    X_valid_cat = X_valid.drop(columns=["product_category"]).copy()

    for col in cat_features:
        X_train_cat[col] = X_train_cat[col].fillna("__MISSING__").astype(str)
        X_valid_cat[col] = X_valid_cat[col].fillna("__MISSING__").astype(str)

    cat_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    cat_model.fit(
        X_train_cat,
        y_train,
        cat_features=cat_features,
    )

    oof_predictions["catboost"][valid_idx] = cat_model.predict(X_valid_cat)

    # Shared preprocessing for XGBoost / LightGBM 
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                SimpleImputer(strategy="median"),
                numeric_features,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OrdinalEncoder(
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                    ),
                ]),
                categorical_features,
            ),
        ]
    )


    #  XGBoost 
    xgb_model = XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
    )

    xgb_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", xgb_model),
    ])

    xgb_pipeline.fit(X_train, y_train)
    oof_predictions["xgboost"][valid_idx] = xgb_pipeline.predict(X_valid)

    
    # LightGBM
    lgbm_model = LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=7,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="regression",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    lgbm_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", lgbm_model),
    ])

    lgbm_pipeline.fit(X_train, y_train)
    oof_predictions["lightgbm"][valid_idx] = lgbm_pipeline.predict(X_valid)


oof_scores = pd.DataFrame({
    model_name: [
        np.sqrt(mean_squared_error(y, predictions))
    ]
    for model_name, predictions in oof_predictions.items()
}).T

oof_scores.columns = ["oof_rmse"]

oof_residuals = pd.DataFrame({
    model_name: y.to_numpy() - predictions
    for model_name, predictions in oof_predictions.items()
})

residual_correlations = oof_residuals.corr()

print("OOF RMSE")
display(oof_scores)

print("\nResidual correlation")
display(residual_correlations)

OOF RMSE


,oof_rmse
catboost,1075.525940
xgboost,1082.560946
lightgbm,1082.164379



Residual correlation


,catboost,xgboost,lightgbm
catboost,1.000000,0.991361,0.993905
xgboost,0.991361,1.000000,0.995317
lightgbm,0.993905,0.995317,1.000000


In [30]:
# Find the best non-negative convex blend of CatBoost, XGBoost, and LightGBM using only locked-fold OOF predictions.

blend_results = []

cat_oof = oof_predictions["catboost"]
xgb_oof = oof_predictions["xgboost"]
lgbm_oof = oof_predictions["lightgbm"]

for cat_weight in np.arange(0.00, 1.01, 0.02):
    for xgb_weight in np.arange(0.00, 1.01 - cat_weight, 0.02):
        lgbm_weight = 1.0 - cat_weight - xgb_weight

        if lgbm_weight < -1e-9:
            continue

        blend_prediction = (
            cat_weight * cat_oof
            + xgb_weight * xgb_oof
            + lgbm_weight * lgbm_oof
        )

        rmse = np.sqrt(
            mean_squared_error(
                y,
                blend_prediction,
            )
        )

        blend_results.append({
            "catboost_weight": cat_weight,
            "xgboost_weight": xgb_weight,
            "lightgbm_weight": lgbm_weight,
            "oof_rmse": rmse,
        })

blend_results = pd.DataFrame(blend_results)

best_blend = (
    blend_results
    .sort_values("oof_rmse")
    .head(10)
    .reset_index(drop=True)
)

best_blend

,catboost_weight,xgboost_weight,lightgbm_weight,oof_rmse
0,0.88,0.12,0.000000e+00,1075.378547
1,0.86,0.14,0.000000e+00,1075.380349
2,0.90,0.10,-2.775558e-17,1075.384280
3,0.84,0.16,2.775558e-17,1075.389685
4,0.92,0.08,-4.163336e-17,1075.397546
5,0.82,0.18,-5.551115e-17,1075.406554
6,0.86,0.12,2.000000e-02,1075.407910
7,0.88,0.10,2.000000e-02,1075.409262
8,0.84,0.14,2.000000e-02,1075.414092
9,0.90,0.08,2.000000e-02,1075.418148


In [31]:
# Aim: Identify where CatBoost's OOF RMSE is concentrated by quantifying absolute and squared error across sales-magnitude groups.

cat_oof = oof_predictions["catboost"]

error_analysis = pd.DataFrame({
    "actual_sales": y.to_numpy(),
    "prediction": cat_oof,
})

error_analysis["error"] = (
    error_analysis["actual_sales"] - error_analysis["prediction"]
)

error_analysis["absolute_error"] = (
    error_analysis["error"].abs()
)

error_analysis["squared_error"] = (
    error_analysis["error"] ** 2
)

error_analysis["sales_decile"] = pd.qcut(
    error_analysis["actual_sales"],
    q=10,
    duplicates="drop",
)

cat_error_by_decile = (
    error_analysis
    .groupby("sales_decile", observed=True)
    .agg(
        observations=("actual_sales", "size"),
        mean_actual=("actual_sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
        total_squared_error=("squared_error", "sum"),
    )
    .reset_index()
)

cat_error_by_decile["squared_error_share_pct"] = (
    100
    * cat_error_by_decile["total_squared_error"]
    / cat_error_by_decile["total_squared_error"].sum()
)

cat_error_by_decile

,sales_decile,observations,mean_actual,mean_prediction,mean_absolute_error,rmse,total_squared_error,squared_error_share_pct
0,"(32.699000000000005, 342.042]",682,191.706393,443.148825,278.912528,456.570289,1.421673e+08,1.802606
1,"(342.042, 669.366]",682,505.351349,1002.726318,553.759166,906.229264,5.600935e+08,7.101691
2,"(669.366, 1018.973]",682,839.744311,1315.385936,579.892275,912.598672,5.679944e+08,7.201870
3,"(1018.973, 1393.5]",681,1205.756990,1782.837931,729.915271,1008.881652,6.931505e+08,8.788784
4,"(1393.5, 1790.89]",682,1592.183871,2052.206786,663.732743,915.021790,5.710146e+08,7.240165
5,"(1790.89, 2256.888]",682,2006.526188,2389.732725,691.207391,885.209146,5.344119e+08,6.776062
6,"(2256.888, 2780.708]",681,2511.836623,2578.374824,618.424736,759.034820,3.923472e+08,4.974755
7,"(2780.708, 3466.5]",682,3104.332698,2970.657992,661.639373,826.484767,4.658586e+08,5.906841
8,"(3466.5, 4557.841]",682,3970.666393,3310.776147,884.015051,1053.936019,7.575527e+08,9.605370
9,"(4557.841, 12996.82]",682,5818.534355,3975.000525,1893.198274,2166.856442,3.202172e+09,40.601855


In [32]:
# Determine whether CatBoost's OOF errors, especially large errors, are concentrated in particular stores.

store_error_analysis = error_analysis.copy()

store_error_analysis["store_code"] = train.loc[
    store_error_analysis.index,
    "store_code",
].to_numpy()

store_error_by_store = (
    store_error_analysis
    .groupby("store_code")
    .agg(
        observations=("actual_sales", "size"),
        mean_actual=("actual_sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_error=("error", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
        total_squared_error=("squared_error", "sum"),
    )
    .reset_index()
)

store_error_by_store["squared_error_share_pct"] = (
    100
    * store_error_by_store["total_squared_error"]
    / store_error_by_store["total_squared_error"].sum()
)

store_error_by_store.sort_values(
    "rmse",
    ascending=False,
)

,store_code,observations,mean_actual,mean_prediction,mean_error,mean_absolute_error,rmse,total_squared_error,squared_error_share_pct
0,STORE-7WS,748,3660.182219,3637.342498,22.839721,1032.716725,1401.568231,1.469366e+09,18.630792
9,STORE-YLW,754,2253.950690,2305.608134,-51.657444,846.910128,1156.462753,1.008404e+09,12.786034
7,STORE-OYG,744,2365.833656,2338.101392,27.732264,835.515038,1150.060890,9.840442e+08,12.477162
2,STORE-9RG,752,2479.168098,2417.831588,61.336510,832.781377,1119.796440,9.429659e+08,11.956312
4,STORE-DKU,736,2177.136481,2245.257419,-68.120938,799.179365,1105.479782,8.994550e+08,11.404615
1,STORE-89Z,728,2365.926745,2322.701637,43.225107,812.377426,1085.929277,8.584885e+08,10.885182
3,STORE-AGY,748,2254.890120,2293.363712,-38.473591,807.098883,1073.861897,8.625782e+08,10.937037
5,STORE-HL7,742,1971.661954,2021.384286,-49.722332,764.489428,1050.818780,8.193313e+08,10.388690
6,STORE-JOR,439,334.435011,351.455857,-17.020846,168.983758,229.150347,2.305184e+07,0.292285
8,STORE-T5G,427,338.326651,345.459130,-7.132479,157.578417,211.370574,1.907730e+07,0.241890


In [33]:
# Determine whether the store-level error concentration is driven by specific product categories.

category_error_analysis = error_analysis.copy()

category_error_analysis["store_code"] = train.loc[
    category_error_analysis.index,
    "store_code",
].to_numpy()

category_error_analysis["product_category_clean"] = train.loc[
    category_error_analysis.index,
    "product_category_clean",
].to_numpy()

category_error_by_store_category = (
    category_error_analysis
    .groupby(["store_code", "product_category_clean"])
    .agg(
        observations=("actual_sales", "size"),
        mean_actual=("actual_sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_error=("error", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
        total_squared_error=("squared_error", "sum"),
    )
    .reset_index()
)

category_error_by_store_category["squared_error_share_pct"] = (
    100
    * category_error_by_store_category["total_squared_error"]
    / category_error_by_store_category["total_squared_error"].sum()
)

category_error_by_store_category.sort_values(
    "total_squared_error",
    ascending=False,
).head(20)

,store_code,product_category_clean,observations,mean_actual,mean_prediction,mean_error,mean_absolute_error,rmse,total_squared_error,squared_error_share_pct
6,STORE-7WS,fruits and vegetables,116,4021.409741,3771.535871,249.873870,1033.491999,1418.557546,2.334274e+08,2.959737
13,STORE-7WS,snack foods,103,3575.135631,3746.087342,-170.951711,1087.207699,1394.281389,2.002341e+08,2.538863
9,STORE-7WS,household,82,3877.592927,3816.886945,60.705982,1090.771490,1559.730598,1.994863e+08,2.529381
54,STORE-AGY,fruits and vegetables,124,2306.836774,2338.986538,-32.149764,857.462272,1184.756059,1.740522e+08,2.206891
22,STORE-89Z,fruits and vegetables,114,2444.613421,2341.495685,103.117736,913.899595,1218.533860,1.692700e+08,2.146255
118,STORE-OYG,fruits and vegetables,103,2558.418932,2465.668201,92.750731,884.859843,1263.297382,1.643798e+08,2.084249
77,STORE-DKU,snack foods,107,2138.355981,2301.938395,-163.582414,894.135779,1220.753604,1.594556e+08,2.021813
4,STORE-7WS,dairy,52,3855.248269,3783.771887,71.476383,1309.257259,1728.174228,1.553025e+08,1.969154
70,STORE-DKU,fruits and vegetables,106,2326.946321,2323.404826,3.541495,850.939711,1201.052831,1.529080e+08,1.938792
45,STORE-9RG,snack foods,111,2604.244414,2441.524563,162.719851,880.332357,1160.321945,1.494445e+08,1.894878


In [34]:
# Inspect the largest CatBoost OOF errors to identify the feature patterns behind the remaining RMSE.

tail_errors = (
    error_analysis
    .join(
        train[
            [
                "product_code",
                "product_weight_kg",
                "fat_content",
                "product_category_clean",
                "product_price",
                "store_code",
                "store_age_years",
                "store_size",
                "store_location_tier",
                "store_format",
            ]
        ]
    )
    .sort_values("absolute_error", ascending=False)
)

tail_errors[
    [
        "actual_sales",
        "prediction",
        "error",
        "absolute_error",
        "product_code",
        "product_category_clean",
        "product_price",
        "product_weight_kg",
        "fat_content",
        "store_code",
        "store_size",
        "store_location_tier",
        "store_format",
    ]
].head(30)

,actual_sales,prediction,error,absolute_error,product_code,product_category_clean,product_price,product_weight_kg,fat_content,store_code,store_size,store_location_tier,store_format
5404,12996.82,5644.635794,7352.184206,7352.184206,PRD-I67HN3,household,232.78,NaN,Low Fat,STORE-7WS,Medium,Tier_3,Flagship Hypermarket
4220,12359.04,5817.634763,6541.405237,6541.405237,PRD-5I78U5,fruits and vegetables,241.26,NaN,Regular,STORE-7WS,Medium,Tier_3,Flagship Hypermarket
900,10130.90,3985.660883,6145.239117,6145.239117,PRD-PUWBWJ,dairy,262.30,12.342,Regular,STORE-OYG,NaN,Tier_2,Standard Supermarket
5950,9999.51,4056.881178,5942.628822,5942.628822,PRD-LV9PLM,health and hygiene,268.58,6.868,Low Fat,STORE-YLW,Small,Tier_1,Standard Supermarket
1555,9388.15,3953.194206,5434.955794,5434.955794,PRD-B9OF3M,fruits and vegetables,262.55,17.089,Low Fat,STORE-DKU,NaN,Tier_2,Standard Supermarket
4511,9072.04,3792.519616,5279.520384,5279.520384,PRD-07NNDG,frozen foods,262.86,9.666,Regular,STORE-DKU,NaN,Tier_2,Standard Supermarket
3444,8617.32,3662.666768,4954.653232,4954.653232,PRD-N5F3DX,snack foods,224.71,16.722,Regular,STORE-OYG,NaN,Tier_2,Standard Supermarket
625,8894.14,3976.191628,4917.948372,4917.948372,PRD-YW8YG4,frozen foods,255.71,5.525,Low Fat,STORE-OYG,NaN,Tier_2,Standard Supermarket
4597,8714.71,3812.342322,4902.367678,4902.367678,PRD-UX01UP,snack foods,242.05,10.675,Low Fat,STORE-DKU,NaN,Tier_2,Standard Supermarket
4363,1377.26,6192.311810,-4815.051810,4815.051810,PRD-6YLBXP,canned,258.91,NaN,Regular,STORE-7WS,Medium,Tier_3,Flagship Hypermarket


In [37]:
#Fix categorical missing values so the product target-encoding experiment can run without changing its modeling hypothesis.

product_te_rmses = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    # Keep CatBoost categorical inputs valid and consistent with the champion pipeline.
    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_valid[col] = X_valid[col].fillna("Missing").astype(str)

    global_mean = y_train.mean()

    product_stats = (
        pd.DataFrame({
            "product_code": X_train["product_code"].values,
            "target": y_train.values,
        })
        .groupby("product_code")["target"]
        .agg(["mean", "count"])
    )

    smoothing = 10

    product_stats["product_te"] = (
        (product_stats["mean"] * product_stats["count"]
         + global_mean * smoothing)
        / (product_stats["count"] + smoothing)
    )

    X_train["product_te"] = (
        X_train["product_code"]
        .map(product_stats["product_te"])
        .fillna(global_mean)
    )

    X_valid["product_te"] = (
        X_valid["product_code"]
        .map(product_stats["product_te"])
        .fillna(global_mean)
    )

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_valid)

    fold_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    product_te_rmses.append(fold_rmse)

    print(f"Fold {fold_num}: {fold_rmse:.4f}")

print(f"\nMean RMSE: {np.mean(product_te_rmses):.4f}")
print(f"Std RMSE: {np.std(product_te_rmses):.4f}")
print(
    f"Change vs champion: "
    f"{1075.3555 - np.mean(product_te_rmses):+.4f}"
)

Fold 1: 1181.6405
Fold 2: 1105.5049
Fold 3: 1189.9296
Fold 4: 1240.7870
Fold 5: 1209.2775

Mean RMSE: 1185.4279
Std RMSE: 44.8458
Change vs champion: -110.0724


In [38]:
# Test whether modestly emphasizing high-sales training observations reduces the upper-tail errors that dominate RMSE.

tail_weight_rmses = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    # Weight observations by their training-fold sales level.
    # sqrt keeps the weighting moderate rather than letting extreme rows dominate.
    train_mean = y_train.mean()
    sample_weight = np.sqrt(y_train / train_mean).clip(lower=0.5, upper=3.0)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        sample_weight=sample_weight,
    )

    pred = model.predict(X_valid)

    fold_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    tail_weight_rmses.append(fold_rmse)

    print(f"Fold {fold_num}: {fold_rmse:.4f}")

tail_weight_mean = np.mean(tail_weight_rmses)
tail_weight_std = np.std(tail_weight_rmses)

print(f"\nMean RMSE: {tail_weight_mean:.4f}")
print(f"Std RMSE: {tail_weight_std:.4f}")
print(f"Change vs champion: {1075.3555 - tail_weight_mean:+.4f}"
      )

Fold 1: 1082.2616
Fold 2: 1078.1458
Fold 3: 1124.8440
Fold 4: 1137.8189
Fold 5: 1087.0489

Mean RMSE: 1102.0238
Std RMSE: 24.4418
Change vs champion: -26.6683


In [39]:
# Test whether an explicit nonlinear price term helps the shallow CatBoost capture curvature in the price-sales relationship.

price_squared_rmses = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    X_train["product_price_squared"] = X_train["product_price"] ** 2
    X_valid["product_price_squared"] = X_valid["product_price"] ** 2

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_valid)

    fold_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    price_squared_rmses.append(fold_rmse)

    print(f"Fold {fold_num}: {fold_rmse:.4f}")

price_squared_mean = np.mean(price_squared_rmses)
price_squared_std = np.std(price_squared_rmses)

print(f"\nMean RMSE: {price_squared_mean:.4f}")
print(f"Std RMSE: {price_squared_std:.4f}")
print(f"Change vs champion: {1075.3555 - price_squared_mean:+.4f}")

Fold 1: 1064.6394
Fold 2: 1052.1159
Fold 3: 1087.1516
Fold 4: 1107.7960
Fold 5: 1065.1078

Mean RMSE: 1075.3621
Std RMSE: 19.7558
Change vs champion: -0.0066


In [40]:
# Determine whether high-sales behavior is concentrated in specific price bands within stores.

price_store_tail = train.copy()

price_store_tail["price_band"] = pd.qcut(
    price_store_tail["product_price"],
    q=5,
    duplicates="drop"
)

price_store_tail["high_sales"] = (
    price_store_tail["total_sales"] >= price_store_tail["total_sales"].quantile(0.90)
)

price_store_tail_profile = (
    price_store_tail
    .groupby(["store_code", "price_band"], observed=True)
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        high_sales_rate=("high_sales", "mean"),
        mean_price=("product_price", "mean"),
    )
    .reset_index()
)

price_store_tail_profile["high_sales_rate"] *= 100

price_store_tail_profile.sort_values(
    "mean_sales", ascending=False
).head(20)

,store_code,price_band,observations,mean_sales,median_sales,high_sales_rate,mean_price
4,STORE-7WS,"(194.228, 273.04]",142,5819.930704,5702.310,70.422535,228.621901
3,STORE-7WS,"(158.83, 194.228]",161,4754.122422,4702.160,53.416149,176.803106
14,STORE-9RG,"(194.228, 273.04]",158,4091.917658,4172.505,37.974684,230.710759
39,STORE-OYG,"(194.228, 273.04]",149,3990.268188,3705.160,36.241611,233.165235
9,STORE-89Z,"(194.228, 273.04]",146,3817.805890,3637.040,31.506849,231.625205
19,STORE-AGY,"(194.228, 273.04]",158,3633.293608,3399.955,28.481013,230.212342
2,STORE-7WS,"(117.848, 158.83]",141,3583.986312,3470.430,20.567376,138.462340
49,STORE-YLW,"(194.228, 273.04]",156,3575.762244,3478.970,29.487179,230.404423
24,STORE-DKU,"(194.228, 273.04]",146,3429.749658,3094.475,23.287671,229.049589
29,STORE-HL7,"(194.228, 273.04]",155,3272.581613,3115.810,21.290323,231.150000


In [ ]:
# Test whether an explicit store × price-band interaction captures the strong nonlinear price behavior seen within stores.

store_price_band_rmses = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_valid[col] = X_valid[col].fillna("Missing").astype(str)

    # Learn price-band boundaries from the training fold only.
    _, price_bins = pd.qcut(
        X_train["product_price"],
        q=5,
        retbins=True,
        duplicates="drop",
    )

    # Ensure validation prices are assigned using the training-fold boundaries.
    X_train["price_band"] = pd.cut(
        X_train["product_price"],
        bins=price_bins,
        include_lowest=True,
    ).astype(str)

    X_valid["price_band"] = pd.cut(
        X_valid["product_price"],
        bins=price_bins,
        include_lowest=True,
    ).astype(str)

    # Explicit store × price-band interaction.
    X_train["store_price_band"] = (
        X_train["store_code"] + "__" + X_train["price_band"]
    )
    X_valid["store_price_band"] = (
        X_valid["store_code"] + "__" + X_valid["price_band"]
    )

    interaction_cat_features = cat_features + [
        "price_band",
        "store_price_band",
    ]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=interaction_cat_features,
    )

    pred = model.predict(X_valid)

    fold_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    store_price_band_rmses.append(fold_rmse)

    print(f"Fold {fold_num}: {fold_rmse:.4f}")

store_price_band_mean = np.mean(store_price_band_rmses)
store_price_band_std = np.std(store_price_band_rmses)

print(f"\nMean RMSE: {store_price_band_mean:.4f}")
print(f"Std RMSE: {store_price_band_std:.4f}")
print(f"Change vs champion: {1075.3555 - store_price_band_mean:+.4f}")

Fold 1: 1074.8842
Fold 2: 1059.3166
Fold 3: 1088.8471
Fold 4: 1123.3688
Fold 5: 1069.8672

Mean RMSE: 1083.2568
Std RMSE: 22.1934
Change vs champion: -7.9013


In [42]:
# Generate leakage-safe OOF predictions from a deliberately different CatBoost configuration to test whether model diversity can improve the blend.

cat_diverse_oof = np.zeros(len(y))

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_valid[col] = X_valid[col].fillna("Missing").astype(str)

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.015,
        depth=5,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    cat_diverse_oof[valid_idx] = model.predict(X_valid)

    print(f"Fold {fold_num} complete")

cat_diverse_rmse = np.sqrt(mean_squared_error(y, cat_diverse_oof))
residual_corr = np.corrcoef(
    y - cat_oof,
    y - cat_diverse_oof
)[0, 1]

print(f"\nDiverse CatBoost OOF RMSE: {cat_diverse_rmse:.4f}")
print(f"Residual correlation vs champion: {residual_corr:.6f}")

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete

Diverse CatBoost OOF RMSE: 1076.0989
Residual correlation vs champion: 0.999394


In [43]:
product_sales_stability = (
    train
    .groupby("product_code")["total_sales"]
    .agg(
        observations="size",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max",
    )
)

product_sales_stability["cv"] = (
    product_sales_stability["std_sales"]
    / product_sales_stability["mean_sales"]
)

product_sales_stability_profile = (
    product_sales_stability
    .query("observations >= 2")
    .groupby("observations")
    .agg(
        products=("mean_sales", "size"),
        median_cv=("cv", "median"),
        mean_cv=("cv", "mean"),
        median_range=("max_sales", lambda x: np.median(x)),
    )
    .reset_index()
)

print(product_sales_stability_profile)

print("\nOverall repeated-product statistics:")
print(
    product_sales_stability
    .query("observations >= 2")[["observations", "cv"]]
    .describe()
)

   observations  products  median_cv   mean_cv  median_range
0             2       140   0.371590  0.477942       2821.89
1             3       251   0.571104  0.569189       3260.23
2             4       394   0.576386  0.575928       3235.06
3             5       351   0.565867  0.573217       3505.48
4             6       240   0.590349  0.586617       3187.50
5             7        97   0.609231  0.606955       3791.41
6             8        35   0.621930  0.646345       4438.53
7             9         1   0.605521  0.605521       2161.90

Overall repeated-product statistics:
       observations           cv
count   1509.000000  1509.000000
mean       4.487740     0.570433
std        1.461491     0.234613
min        2.000000     0.002935
25%        3.000000     0.414485
50%        4.000000     0.573912
75%        5.000000     0.710269
max        9.000000     1.466994


In [44]:
# Quantify whether repeated-product sales variability is substantially explained by the store, which determines whether product×store interactions are worth testing.

product_store_sales = (
    train
    .groupby(["product_code", "store_code"])["total_sales"]
    .agg(
        observations="size",
        mean_sales="mean",
        std_sales="std",
    )
    .reset_index()
)

product_store_sales["cv"] = (
    product_store_sales["std_sales"]
    / product_store_sales["mean_sales"]
)

print("Product-store combinations:", len(product_store_sales))
print("Repeated product-store combinations:", (product_store_sales["observations"] >= 2).sum())

print("\nProduct-store CV summary:")
print(
    product_store_sales.loc[
        product_store_sales["observations"] >= 2, "cv"
    ].describe()
)

print("\nProduct observations by number of stores:")
print(
    train.groupby("product_code")["store_code"]
    .nunique()
    .value_counts()
    .sort_index()
)

Product-store combinations: 6818
Repeated product-store combinations: 0

Product-store CV summary:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: cv, dtype: float64

Product observations by number of stores:
store_code
1     46
2    140
3    251
4    394
5    351
6    240
7     97
8     35
9      1
Name: count, dtype: int64


In [45]:
# Test whether CatBoost's ordered boosting improves generalization for our categorical-heavy dataset.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

ordered_scores = []

for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_val = X_cat.iloc[val_idx].copy()
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_val[col] = X_val[col].fillna("Missing").astype(str)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        boosting_type="Ordered",
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    ordered_scores.append(rmse)

    print(f"Fold {fold}: {rmse:.4f}")

print(f"\nOrdered boosting CV RMSE: {np.mean(ordered_scores):.4f}")
print(f"Std: {np.std(ordered_scores):.4f}")
print(
    f"Change vs champion: "
    f"{1075.3555 - np.mean(ordered_scores):+.4f}"
)

Fold 1: 1065.3577
Fold 2: 1052.8215
Fold 3: 1085.5682
Fold 4: 1108.2951
Fold 5: 1062.8881

Ordered boosting CV RMSE: 1074.9861
Std: 19.7576
Change vs champion: +0.3694


In [46]:
# Aim: Test whether CatBoost's L2 leaf regularization can improve the current Ordered boosting champion.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

l2_results = []

for l2_value in [1, 3, 5, 10, 20]:
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(folds, start=1):
        X_train = X_cat.iloc[train_idx].copy()
        X_val = X_cat.iloc[val_idx].copy()
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        for col in cat_features:
            X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
            X_val[col] = X_val[col].fillna("__MISSING__").astype(str)

        model = CatBoostRegressor(
            iterations=300,
            learning_rate=0.025,
            depth=4,
            l2_leaf_reg=l2_value,
            loss_function="RMSE",
            random_seed=42,
            boosting_type="Ordered",
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
        )

        pred = model.predict(X_val)
        fold_scores.append(
            np.sqrt(mean_squared_error(y_val, pred))
        )

    l2_results.append({
        "l2_leaf_reg": l2_value,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

l2_results = pd.DataFrame(l2_results)

print(l2_results)
print("\nBest:")
print(l2_results.loc[l2_results["mean_rmse"].idxmin()])

   l2_leaf_reg    mean_rmse   std_rmse
0            1  1074.915874  19.310738
1            3  1074.986121  19.757582
2            5  1075.288297  19.431896
3           10  1075.559436  19.385734
4           20  1075.289631  18.660674

Best:
l2_leaf_reg       1.000000
mean_rmse      1074.915874
std_rmse         19.310738
Name: 0, dtype: float64


In [48]:
# Aim: Measure high-sales-tail rates across categorical groups without changing the modeling dataset.

train["_sales_tail"] = tail_flag

for col in [
    "product_category_clean",
    "store_code",
    "fat_content",
    "store_size",
    "store_location_tier",
    "store_format",
]:
    profile = (
        train.groupby(col)
        .agg(
            observations=("total_sales", "size"),
            mean_sales=("total_sales", "mean"),
            tail_rate=("_sales_tail", "mean"),
        )
        .sort_values("tail_rate", ascending=False)
    )

    print(f"\n--- {col} ---")
    print(profile)

train.drop(columns="_sales_tail", inplace=True)


--- product_category_clean ---
                        observations   mean_sales  tail_rate
product_category_clean                                      
seafood                           52  2262.524231   0.134615
starchy foods                    113  2407.718584   0.123894
meat                             339  2264.310383   0.120944
fruits and vegetables           1006  2277.854672   0.119284
dairy                            554  2238.317076   0.117329
household                        740  2292.464662   0.106757
frozen foods                     675  2140.770133   0.106667
snack foods                      933  2223.830568   0.102894
canned                           517  2196.398743   0.094778
soft drinks                      366  1967.399454   0.087432
breads                           209  2186.152871   0.086124
others                           132  1947.435833   0.075758
baking goods                     516  1943.416860   0.073643
breakfast                         85  1852.197765   0

In [49]:
# Test whether a row's price relative to its store's typical price adds signal beyond raw price and store identity.

store_relative_scores = []

for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_val = X_cat.iloc[val_idx].copy()
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    train_price = train.iloc[train_idx].copy()
    val_price = train.iloc[val_idx].copy()

    store_median_price = (
        train_price
        .groupby("store_code")["product_price"]
        .median()
    )

    global_median_price = train_price["product_price"].median()

    train_store_median = (
        train_price["store_code"]
        .map(store_median_price)
        .fillna(global_median_price)
    )

    val_store_median = (
        val_price["store_code"]
        .map(store_median_price)
        .fillna(global_median_price)
    )

    X_train["store_relative_price"] = (
        train_price["product_price"] / train_store_median
    )

    X_val["store_relative_price"] = (
        val_price["product_price"] / val_store_median
    )

    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_val[col] = X_val[col].fillna("__MISSING__").astype(str)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        loss_function="RMSE",
        random_seed=42,
        boosting_type="Ordered",
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    store_relative_scores.append(rmse)

    print(f"Fold {fold}: {rmse:.4f}")

print(f"\nStore-relative price CV RMSE: {np.mean(store_relative_scores):.4f}")
print(f"Std: {np.std(store_relative_scores):.4f}")
print(
    f"Change vs current champion: "
    f"{1074.915874 - np.mean(store_relative_scores):+.4f}"
)

Fold 1: 1066.7600
Fold 2: 1054.3620
Fold 3: 1086.8134
Fold 4: 1109.7584
Fold 5: 1061.7661

Store-relative price CV RMSE: 1075.8920
Std: 20.0633
Change vs current champion: -0.9761


In [50]:
# Determine whether extreme prediction errors are concentrated among products with many or few observations.

error_product_frequency = error_analysis.copy()

product_counts = (
    train["product_code"]
    .value_counts()
    .rename("product_observations")
)

error_product_frequency["product_observations"] = (
    train.loc[error_product_frequency.index, "product_code"]
    .map(product_counts)
)

error_product_frequency["error_direction"] = np.where(
    error_product_frequency["error"] < 0,
    "underprediction",
    "overprediction",
)

error_product_frequency["error_magnitude"] = (
    error_product_frequency["absolute_error"]
)

print(
    error_product_frequency.groupby("product_observations")
    .agg(
        rows=("error", "size"),
        mean_abs_error=("error_magnitude", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(x.mean())),
        mean_error=("error", "mean"),
    )
    .reset_index()
)

print("\nExtreme errors:")
print(
    error_product_frequency[
        error_product_frequency["error_magnitude"]
        >= error_product_frequency["error_magnitude"].quantile(0.90)
    ][
        [
            "actual_sales",
            "prediction",
            "error",
            "absolute_error",
            "product_observations",
            "sales_decile",
        ]
    ]
    .sort_values("absolute_error", ascending=False)
    .head(20)
)

   product_observations  rows  mean_abs_error         rmse  mean_error
0                     1    46      809.584539  1181.023378 -161.349428
1                     2   280      822.208855  1132.137927   10.988473
2                     3   753      798.939544  1138.891493    6.685142
3                     4  1576      748.044501  1064.271169  -31.129113
4                     5  1755      776.923898  1093.716677    1.334467
5                     6  1440      694.778122  1021.483947  -16.957537
6                     7   679      757.898667  1059.397987   32.055875
7                     8   280      793.477297  1096.803865  -10.254613
8                     9     9      245.236499   355.231318  176.253066

Extreme errors:
      actual_sales   prediction        error  absolute_error  \
5404      12996.82  5644.635794  7352.184206     7352.184206   
4220      12359.04  5817.634763  6541.405237     6541.405237   
900       10130.90  3985.660883  6145.239117     6145.239117   
5950       9999.5

In [51]:
# Measure whether price remains informative within each sales decile, especially inside the high-sales tail.

price_sales_by_decile = (
    train.assign(
        sales_decile=pd.qcut(
            train["total_sales"],
            q=10,
            duplicates="drop"
        )
    )
    .groupby("sales_decile", observed=True)
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        mean_price=("product_price", "mean"),
        median_price=("product_price", "median"),
        price_sales_corr=(
            "product_price",
            lambda x: x.corr(
                train.loc[x.index, "total_sales"]
            )
        ),
    )
    .reset_index()
)

print(price_sales_by_decile)

                    sales_decile  observations   mean_sales  mean_price  \
0  (32.699000000000005, 342.042]           682   191.706393  102.666144   
1             (342.042, 669.366]           682   505.351349  108.053196   
2            (669.366, 1018.973]           682   839.744311   97.681276   
3             (1018.973, 1393.5]           681  1205.756990  112.234185   
4              (1393.5, 1790.89]           682  1592.183871  124.214795   
5            (1790.89, 2256.888]           682  2006.526188  141.783123   
6           (2256.888, 2780.708]           681  2511.836623  152.290587   
7             (2780.708, 3466.5]           682  3104.332698  170.356202   
8             (3466.5, 4557.841]           682  3970.666393  185.525777   
9           (4557.841, 12996.82]           682  5818.534355  209.906862   

   median_price  price_sales_corr  
0        95.460          0.269089  
1        96.070         -0.151661  
2        80.115          0.044071  
3       102.930          0.125

In [52]:
# Identify which store-category groups dominate the high-sales tail and whether their tail behavior is unusually strong.

tail_cutoff = train["total_sales"].quantile(0.90)

tail_store_category = (
    train.loc[train["total_sales"] >= tail_cutoff]
    .groupby(["store_code", "product_category_clean"])
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        mean_price=("product_price", "mean"),
        max_sales=("total_sales", "max"),
    )
    .reset_index()
)

tail_store_category = tail_store_category.sort_values(
    ["mean_sales", "observations"],
    ascending=[False, False]
)

print(tail_store_category.head(25).to_string(index=False))

store_code product_category_clean  observations  mean_sales  median_sales  mean_price  max_sales
 STORE-AGY            hard drinks             1 8141.490000      8141.490  251.080000    8141.49
 STORE-YLW     health and hygiene             4 7612.530000      7860.975  230.840000    9999.51
 STORE-89Z              breakfast             1 7125.940000      7125.940  233.830000    7125.94
 STORE-7WS                  dairy            14 7099.707143      6393.600  205.020714   10477.97
 STORE-7WS            soft drinks            10 7027.707000      7085.070  185.966000    9803.38
 STORE-89Z     health and hygiene             1 6941.160000      6941.160  195.470000    6941.16
 STORE-HL7                 canned             2 6790.135000      6790.135  261.060000    7016.22
 STORE-DKU           frozen foods             6 6678.790000      6522.375  244.323333    9072.04
 STORE-OYG           baking goods             1 6670.190000      6670.190  258.440000    6670.19
 STORE-AGY            soft dri

In [53]:
# Test whether high-sales tail behavior within store-category groups is primarily associated with price.

tail_store_category_price = (
    train.loc[train["total_sales"] >= tail_cutoff]
    .groupby(["store_code", "product_category_clean"])
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        mean_price=("product_price", "mean"),
        min_price=("product_price", "min"),
        max_price=("product_price", "max"),
        price_sales_corr=(
            "product_price",
            lambda x: x.corr(train.loc[x.index, "total_sales"])
        ),
    )
    .reset_index()
)

tail_store_category_price = tail_store_category_price.sort_values(
    ["observations", "mean_sales"],
    ascending=[False, False]
)

print(
    tail_store_category_price[
        tail_store_category_price["observations"] >= 5
    ].to_string(index=False)
)

c:\Users\Administrator\Documents\python training and projects\env\lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\Administrator\Documents\python training and projects\env\lib\site-packages\numpy\lib\_function_base_impl.py:2894: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\Administrator\Documents\python training and projects\env\lib\site-packages\numpy\lib\_function_base_impl.py:2894: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


store_code product_category_clean  observations  mean_sales  mean_price  min_price  max_price  price_sales_corr
 STORE-7WS  fruits and vegetables            43 6135.629070  187.834186     110.48     266.80          0.572064
 STORE-7WS            snack foods            31 5779.822581  186.356774     111.76     258.26          0.339312
 STORE-7WS              household            30 6289.519667  199.084000     121.00     265.65          0.408024
 STORE-7WS           frozen foods            23 6363.699565  211.018261     134.30     268.75          0.125537
 STORE-7WS                   meat            17 6500.438235  200.357647     101.33     256.98          0.510455
 STORE-7WS                 canned            16 5997.521875  193.353750     101.51     260.28          0.461039
 STORE-89Z  fruits and vegetables            15 5425.553333  201.605333     140.90     255.00          0.762566
 STORE-7WS                  dairy            14 7099.707143  205.020714     151.19     257.16          0

In [54]:
# Determine whether CatBoost systematically underpredicts or overpredicts high-sales observations at different price levels.

tail_error_price = error_analysis.copy()

tail_error_price["price"] = train.loc[tail_error_price.index, "product_price"]
tail_error_price["tail_flag"] = (
    tail_error_price["actual_sales"] >= tail_cutoff
)

tail_error_price = tail_error_price.loc[
    tail_error_price["tail_flag"]
].copy()

tail_error_price["price_bin"] = pd.qcut(
    tail_error_price["price"],
    q=5,
    duplicates="drop"
)

tail_price_error_profile = (
    tail_error_price
    .groupby("price_bin", observed=True)
    .agg(
        observations=("error", "size"),
        mean_price=("price", "mean"),
        mean_actual=("actual_sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_error=("error", "mean"),
        mean_abs_error=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(x.mean())),
    )
    .reset_index()
)

print(tail_price_error_profile.to_string(index=False))

         price_bin  observations  mean_price  mean_actual  mean_prediction  mean_error  mean_abs_error        rmse
(101.329, 177.504]           137  156.727737  5365.294891      3403.303750 1961.991140     1965.893030 2152.577917
 (177.504, 195.11]           136  186.498750  5757.429485      3843.084701 1914.344784     1942.386083 2183.948095
 (195.11, 225.996]           136  212.006618  5755.714706      3939.133306 1816.581400     1834.835453 2096.396603
 (225.996, 248.77]           136  236.773750  6140.917059      4344.257990 1796.659069     1898.089746 2232.432575
  (248.77, 271.82]           137  257.568029  6074.764234      4346.693491 1728.070742     1824.755793 2166.779927


In [55]:
# Test whether CatBoost split-score randomization improves the current Ordered CatBoost champion under the locked 5-fold CV.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

random_strength_values = [0, 0.25, 0.5, 1.0, 2.0]

random_strength_results = []

for random_strength in random_strength_values:
    fold_scores = []

    for train_idx, valid_idx in folds:
        X_train = X_cat.iloc[train_idx].copy()
        X_valid = X_cat.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
            X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

        model = CatBoostRegressor(
            iterations=300,
            learning_rate=0.025,
            depth=4,
            l2_leaf_reg=1,
            boosting_type="Ordered",
            random_strength=random_strength,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
        )

        pred = model.predict(X_valid)
        rmse = np.sqrt(mean_squared_error(y_valid, pred))
        fold_scores.append(rmse)

    random_strength_results.append({
        "random_strength": random_strength,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

random_strength_results = pd.DataFrame(random_strength_results)

print(random_strength_results)

   random_strength    mean_rmse   std_rmse
0             0.00  1076.135342  20.304341
1             0.25  1075.946831  20.400768
2             0.50  1075.038402  19.712375
3             1.00  1074.915874  19.310738
4             2.00  1076.265941  19.215002


In [56]:
# Verify the tail threshold and class balance before designing the leakage-safe two-stage regime model.

tail_cutoff = y.quantile(0.90)
tail_flag = (y >= tail_cutoff).astype(int)

print(f"Tail cutoff (90th percentile): {tail_cutoff:.6f}")
print(f"Tail rows: {tail_flag.sum()} / {len(y)} ({tail_flag.mean():.2%})")

print("\nTail rate by locked fold:")
for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(
        f"Fold {fold_num}: "
        f"train={tail_flag.iloc[train_idx].mean():.2%}, "
        f"valid={tail_flag.iloc[valid_idx].mean():.2%}"
    )

Tail cutoff (90th percentile): 4557.841000
Tail rows: 682 / 6818 (10.00%)

Tail rate by locked fold:
Fold 1: train=9.90%, valid=10.41%
Fold 2: train=10.18%, valid=9.31%
Fold 3: train=10.18%, valid=9.31%
Fold 4: train=9.94%, valid=10.27%
Fold 5: train=9.83%, valid=10.71%


In [57]:
# Test a leakage-safe two-stage tail model against the current Ordered CatBoost champion using the locked 5-fold CV.

from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

tail_cutoff = y.quantile(0.90)
tail_models_results = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    # Create the tail label from training-fold targets only.
    y_tail_train = (y_train >= tail_cutoff).astype(int)

    # Prepare categorical features exactly as in the champion.
    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    # Stage 1: classify whether an observation belongs to the high-sales tail.
    classifier = CatBoostClassifier(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        loss_function="Logloss",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    classifier.fit(
        X_train,
        y_tail_train,
        cat_features=cat_features,
    )

    tail_probability = classifier.predict_proba(X_valid)[:, 1]

    # Stage 2a: model non-tail sales.
    non_tail_mask = y_train < tail_cutoff

    non_tail_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    non_tail_model.fit(
        X_train.loc[non_tail_mask],
        y_train.loc[non_tail_mask],
        cat_features=cat_features,
    )

    # Stage 2b: model tail sales.
    tail_mask = y_train >= tail_cutoff

    tail_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    tail_model.fit(
        X_train.loc[tail_mask],
        y_train.loc[tail_mask],
        cat_features=cat_features,
    )

    non_tail_pred = non_tail_model.predict(X_valid)
    tail_pred = tail_model.predict(X_valid)

    # Soft regime combination avoids a hard classifier boundary.
    pred = (
        (1 - tail_probability) * non_tail_pred
        + tail_probability * tail_pred
    )

    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    tail_models_results.append({
        "fold": fold_num,
        "rmse": rmse,
        "validation_tail_rate": (y_valid >= tail_cutoff).mean(),
    })

tail_models_results = pd.DataFrame(tail_models_results)

print(tail_models_results.to_string(index=False))
print(f"\nMean CV RMSE: {tail_models_results['rmse'].mean():.6f}")
print(f"Std CV RMSE:  {tail_models_results['rmse'].std():.6f}")
print("Champion CV RMSE: 1074.915874")

 fold        rmse  validation_tail_rate
    1 1071.340663              0.104106
    2 1053.370274              0.093109
    3 1082.466843              0.093109
    4 1109.138003              0.102715
    5 1059.616649              0.107117

Mean CV RMSE: 1075.186487
Std CV RMSE:  22.016280
Champion CV RMSE: 1074.915874


In [58]:
# Reproduce the final Ordered CatBoost champion across the locked folds before freezing the modeling decision.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

final_confirmation = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    final_confirmation.append({
        "fold": fold_num,
        "rmse": rmse,
    })

final_confirmation = pd.DataFrame(final_confirmation)

print(final_confirmation.to_string(index=False))
print(f"\nMean CV RMSE: {final_confirmation['rmse'].mean():.6f}")
print(f"Std CV RMSE:  {final_confirmation['rmse'].std():.6f}")
print("Frozen benchmark: 1074.915874")

 fold        rmse
    1 1065.990810
    2 1053.887357
    3 1085.198997
    4 1107.592529
    5 1061.909675

Mean CV RMSE: 1074.915874
Std CV RMSE:  21.590061
Frozen benchmark: 1074.915874


In [59]:
# Test whether price_per_kg provides incremental out-of-fold predictive value beyond the frozen CatBoost benchmark.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

X_fe = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

X_fe["price_per_kg"] = (
    X_fe["product_price"] / X_fe["product_weight_kg"]
)

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for col in cat_features:
    X_fe[col] = X_fe[col].fillna("__MISSING__").astype(str)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_fe.iloc[train_idx]
    X_valid = X_fe.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    fold_scores.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print(f"\nMean CV RMSE: {np.mean(fold_scores):.4f}")
print(f"Std CV RMSE:  {np.std(fold_scores):.4f}")
print(f"Frozen benchmark: 1074.9159")
print(f"Change: {1074.9159 - np.mean(fold_scores):+.4f}")

Fold 1: RMSE = 1066.7261
Fold 2: RMSE = 1053.6681
Fold 3: RMSE = 1086.9136
Fold 4: RMSE = 1109.3689
Fold 5: RMSE = 1062.9534

Mean CV RMSE: 1075.9260
Std CV RMSE:  19.9374
Frozen benchmark: 1074.9159
Change: -1.0101
